# Model Improvement and Final Model Selection

## Objective

This notebook extends the baseline multi-label recommendation experiments and
investigates why model performance decreases substantially when environmental
weather groups are held out.

The main objectives are:

1. Analyse recommendation performance separately for each target label.
2. Identify labels that depend strongly on a small number of weather contexts.
3. Examine feature redundancy and indirect relationships with rule-derived
   targets.
4. Compare model behaviour under stricter evaluation conditions.
5. Select an appropriate model for the context-aware offline recommendation
   prototype.
6. Consider computational and storage requirements for eventual mobile
   deployment.

Model selection will not be based solely on the high random-split score.
Generalization behaviour, interpretability, model complexity, and mobile
deployment suitability will also be considered.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    accuracy_score
)

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "research").is_dir()
        and (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"
DATASET_DIR = RESEARCH_DIR / "dataset"

FEATURE_FILE = (
    DATASET_DIR
    / "processed"
    / "features"
    / "knuckles_trail_context_weather_features.csv"
)

LABEL_FILE = (
    DATASET_DIR
    / "labels"
    / "knuckles_offline_resource_labels.csv"
)

RESULTS_DIR = RESEARCH_DIR / "results"
MODELS_DIR = RESEARCH_DIR / "models"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("MODEL IMPROVEMENT ENVIRONMENT")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("Feature dataset:", FEATURE_FILE.exists())
print("Label dataset:", LABEL_FILE.exists())

MODEL IMPROVEMENT ENVIRONMENT
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
Feature dataset: True
Label dataset: True


In [2]:
features = pd.read_csv(
    FEATURE_FILE
)

labels = pd.read_csv(
    LABEL_FILE
)

TARGET_COLUMNS = [
    "hypothermia_guidance",
    "heat_illness_guidance",
    "heavy_rain_guidance",
    "strong_wind_guidance",
    "water_safety_guidance",
    "remote_area_emergency_guidance"
]

model_data = features.merge(
    labels[
        ["id"] + TARGET_COLUMNS
    ],
    on="id",
    how="inner",
    validate="one_to_one"
)

print("DATASET VALIDATION")
print("=" * 60)

print("Features:", features.shape)
print("Labels:", labels.shape)
print("Merged:", model_data.shape)

print(
    "Duplicate IDs:",
    model_data["id"].duplicated().sum()
)

print(
    "Missing values:",
    model_data.isna().sum().sum()
)

DATASET VALIDATION
Features: (557, 49)
Labels: (557, 8)
Merged: (557, 55)
Duplicate IDs: 0
Missing values: 0


In [3]:
group_label_distribution = (
    model_data
    .groupby("point_id")[
        TARGET_COLUMNS
    ]
    .agg(["sum", "mean"])
)

print("LABEL DISTRIBUTION BY WEATHER GROUP")
print("=" * 60)

display(
    group_label_distribution.round(3)
)

LABEL DISTRIBUTION BY WEATHER GROUP


hypothermia_guidance        heat_illness_guidance       \
                            sum   mean                   sum mean   
point_id                                                            
knuckles_0                  104  0.788                     0  0.0   
knuckles_1                    0  0.000                   257  1.0   
knuckles_2                    0  0.000                     0  0.0   
knuckles_3                   19  0.257                     0  0.0   
knuckles_4                    0  0.000                    26  1.0   

           heavy_rain_guidance      strong_wind_guidance       \
                           sum mean                  sum mean   
point_id                                                        
knuckles_0                   0  0.0                  132  1.0   
knuckles_1                 257  1.0                    0  0.0   
knuckles_2                   0  0.0                    0  0.0   
knuckles_3                   0  0.0                   74  1.0   
knuckles_4                   0  0.0                    0  0.0   

           water_safety_guidance        remote_area_emergency_guidance         
                             sum   mean                            sum   mean  
point_id                                                                       
knuckles_0                    11  0.083                             59  0.447  
knuckles_1                    82  0.319                             62  0.241  
knuckles_2                    10  0.147                             26  0.382  
knuckles_3                    11  0.149                             74  1.000  
knuckles_4                    26  1.000                             26  1.000

In [4]:
group_positive_counts = (
    model_data
    .groupby("point_id")[
        TARGET_COLUMNS
    ]
    .sum()
    .astype(int)
)

group_sizes = (
    model_data
    .groupby("point_id")
    .size()
    .rename("trail_count")
)

group_positive_counts = (
    group_sizes.to_frame()
    .join(group_positive_counts)
)

print("POSITIVE TARGETS BY WEATHER GROUP")
print("=" * 60)

display(
    group_positive_counts
)

POSITIVE TARGETS BY WEATHER GROUP


,trail_count,hypothermia_guidance,heat_illness_guidance,heavy_rain_guidance,strong_wind_guidance,water_safety_guidance,remote_area_emergency_guidance
point_id,,,,,,,
knuckles_0,132,104,0,0,132,11,59
knuckles_1,257,0,257,257,0,82,62
knuckles_2,68,0,0,0,0,10,26
knuckles_3,74,19,0,0,74,11,74
knuckles_4,26,0,26,0,0,26,26


## Weather-Group Dependency Analysis

Analysis of target distributions reveals strong dependence between several
recommendation labels and the five historical weather reference locations.

In particular:

- `heat_illness_guidance` is positive for all trails assigned to knuckles_1
  and knuckles_4 and negative for the other weather groups.
- `heavy_rain_guidance` is positive only for trails assigned to knuckles_1.
- `strong_wind_guidance` is positive for all trails assigned to knuckles_0
  and knuckles_3 and negative for the remaining groups.

Consequently, these targets currently contain little or no within-weather-group
variation. A model trained on trail-level rows can therefore learn associations
with regional weather profiles rather than generalizable trail-specific hazard
relationships.

The terrain/remoteness-related targets show greater within-group variation,
although some groups still contain highly concentrated positive labels.

This analysis confirms that increasing model complexity alone cannot resolve
the generalization limitation. Broader environmental context and independently
validated target labels are required for stronger future evaluation.

In [5]:
within_group_variation = []

for label in TARGET_COLUMNS:

    mixed_groups = 0
    constant_groups = 0

    for _, group_df in model_data.groupby(
        "point_id"
    ):
        unique_values = (
            group_df[label]
            .nunique()
        )

        if unique_values > 1:
            mixed_groups += 1
        else:
            constant_groups += 1

    within_group_variation.append({
        "label": label,
        "mixed_groups": mixed_groups,
        "constant_groups": constant_groups,
        "total_groups": model_data[
            "point_id"
        ].nunique()
    })

within_group_variation = pd.DataFrame(
    within_group_variation
)

within_group_variation[
    "mixed_group_percent"
] = (
    within_group_variation[
        "mixed_groups"
    ]
    /
    within_group_variation[
        "total_groups"
    ]
    * 100
)

print("WITHIN-GROUP TARGET VARIATION")
print("=" * 60)

display(
    within_group_variation
)

WITHIN-GROUP TARGET VARIATION


,label,mixed_groups,constant_groups,total_groups,mixed_group_percent
0,hypothermia_guidance,2,3,5,40.0
1,heat_illness_guidance,0,5,5,0.0
2,heavy_rain_guidance,0,5,5,0.0
3,strong_wind_guidance,0,5,5,0.0
4,water_safety_guidance,4,1,5,80.0
5,remote_area_emergency_guidance,3,2,5,60.0


In [6]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):

    table = pd.crosstab(
        x,
        y
    )

    chi2 = chi2_contingency(
        table,
        correction=False
    )[0]

    n = table.to_numpy().sum()

    phi2 = chi2 / n

    r, k = table.shape

    denominator = min(
        k - 1,
        r - 1
    )

    if denominator == 0:
        return np.nan

    return np.sqrt(
        phi2 / denominator
    )


group_association = []

for label in TARGET_COLUMNS:

    association = cramers_v(
        model_data["point_id"],
        model_data[label]
    )

    group_association.append({
        "label": label,
        "cramers_v_with_weather_group":
            association
    })

group_association = pd.DataFrame(
    group_association
).sort_values(
    "cramers_v_with_weather_group",
    ascending=False
)

print("WEATHER GROUP / TARGET ASSOCIATION")
print("=" * 60)

display(
    group_association.round(4)
)

WEATHER GROUP / TARGET ASSOCIATION


,label,cramers_v_with_weather_group
1,heat_illness_guidance,1.0000
2,heavy_rain_guidance,1.0000
3,strong_wind_guidance,1.0000
0,hypothermia_guidance,0.7890
5,remote_area_emergency_guidance,0.5510
4,water_safety_guidance,0.4476


In [7]:
target_characteristics = pd.DataFrame({
    "target": TARGET_COLUMNS,

    "primary_context": [
        "weather + terrain",
        "weather",
        "weather",
        "weather",
        "resource proximity",
        "emergency/remoteness"
    ],

    "current_weather_group_dependency": [
        "high",
        "very high",
        "very high",
        "very high",
        "lower",
        "lower"
    ],

    "prototype_role": [
        "contextual",
        "contextual",
        "contextual",
        "contextual",
        "contextual",
        "contextual"
    ]
})

print("TARGET CHARACTERISTICS")
print("=" * 60)

display(
    target_characteristics
)

TARGET CHARACTERISTICS


,target,primary_context,current_weather_group_dependency,prototype_role
0,hypothermia_guidance,weather + terrain,high,contextual
1,heat_illness_guidance,weather,very high,contextual
2,heavy_rain_guidance,weather,very high,contextual
3,strong_wind_guidance,weather,very high,contextual
4,water_safety_guidance,resource proximity,lower,contextual
5,remote_area_emergency_guidance,emergency/remoteness,lower,contextual


## Model-Improvement Decision

The analysis indicates that the primary limitation is data representation
rather than insufficient model complexity.

Therefore, the next development stage will not attempt to maximize the
random-split score through aggressive hyperparameter tuning.

Instead, the prototype will use the following strategy:

1. Retain the six-label multi-label formulation as the research prototype.
2. Retain both random-split and grouped evaluations for transparent reporting.
3. Treat grouped evaluation as the stronger test of regional generalization.
4. Prefer simpler and more interpretable models when performance is comparable.
5. Keep mandatory safety resources outside the ML prediction layer.
6. Treat weather-related recommendations conservatively because the current
   historical weather representation contains only five spatial profiles.
7. Design the deployed system so that current or forecast weather can later
   replace or complement historical regional weather context.
8. Identify broader weather coverage and independent expert validation as
   required future improvements.

The current research stage therefore focuses on selecting a defensible
prototype model rather than optimizing performance against rule-derived labels.

In [8]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Random_Split_Macro_F1": [
        0.9444,
        0.9613
    ],

    "Grouped_Mean_Macro_F1": [
        0.2944,
        0.2344
    ],

    "Random_Split_Hamming_Loss": [
        0.0357,
        0.0238
    ],

    "Grouped_Mean_Hamming_Loss": [
        0.2762,
        0.3040
    ],

    "Interpretability": [
        "High",
        "Medium"
    ],

    "Model_Complexity": [
        "Low",
        "High"
    ],

    "Mobile_Deployment_Suitability": [
        "High",
        "Medium"
    ]
})

final_model_comparison[
    "Generalization_F1_Drop"
] = (
    final_model_comparison[
        "Random_Split_Macro_F1"
    ]
    -
    final_model_comparison[
        "Grouped_Mean_Macro_F1"
    ]
)

print("FINAL MODEL COMPARISON")
print("=" * 60)

display(
    final_model_comparison.round(4)
)

FINAL MODEL COMPARISON


,Model,Random_Split_Macro_F1,Grouped_Mean_Macro_F1,Random_Split_Hamming_Loss,Grouped_Mean_Hamming_Loss,Interpretability,Model_Complexity,Mobile_Deployment_Suitability,Generalization_F1_Drop
0,Logistic Regression,0.9444,0.2944,0.0357,0.2762,High,Low,High,0.6500
1,Random Forest,0.9613,0.2344,0.0238,0.3040,Medium,High,Medium,0.7269


## Final Prototype Model Selection

Logistic Regression is selected as the primary prototype model for the
context-aware offline resource recommendation component.

Although Random Forest achieved the highest random trail-level split macro
F1-score (0.9613 compared with 0.9444), this random-split result is not the
primary basis for final model selection.

Under the stricter Leave-One-Weather-Group-Out evaluation, Logistic Regression
achieved a higher mean macro F1-score:

- Logistic Regression: 0.2944
- Random Forest: 0.2344

Logistic Regression also achieved a lower grouped Hamming Loss:

- Logistic Regression: 0.2762
- Random Forest: 0.3040

Therefore, Logistic Regression demonstrated comparatively better
generalization when an environmental weather group was excluded from training.

It also provides practical advantages for the intended mobile application:

- lower model complexity;
- greater interpretability;
- smaller expected storage requirements;
- lower inference computational cost;
- easier implementation of offline inference.

The selected model remains a research prototype. Its grouped performance shows
that additional spatial weather coverage and independently validated labels are
required before making claims about real-world safety effectiveness.

In [9]:
DIRECT_LABEL_RULE_FEATURES = [
    "hist_temperature_p05_c",
    "elevation_gain_m",
    "maximum_elevation_m",
    "hist_apparent_temperature_p95_c",
    "hist_precipitation_p95_mm",
    "hist_rain_hour_percent",
    "hist_wind_gust_p95_kmh",
    "nearest_drinking_water_km",
    "nearest_hospital_km",
    "nearest_ranger_station_km",
    "nearest_communication_structure_km"
]

EXCLUDED_MODEL_COLUMNS = (
    ["id"]
    + TARGET_COLUMNS
    + DIRECT_LABEL_RULE_FEATURES
    + [
        "point_id",
        "hist_rain_hour_fraction"
    ]
)

MODEL_FEATURE_COLUMNS = [
    column
    for column in model_data.columns
    if column not in EXCLUDED_MODEL_COLUMNS
]

X = model_data[
    MODEL_FEATURE_COLUMNS
].copy()

y = model_data[
    TARGET_COLUMNS
].copy()

categorical_features = (
    X.select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_features = (
    X.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)

print("DEPLOYMENT FEATURE SET")
print("=" * 60)

print(
    "Total features:",
    len(MODEL_FEATURE_COLUMNS)
)

print(
    "Numeric:",
    len(numeric_features)
)

print(
    "Categorical:",
    categorical_features
)

print(
    "Targets:",
    len(TARGET_COLUMNS)
)

DEPLOYMENT FEATURE SET
Total features: 35
Numeric: 34
Categorical: ['highway']
Targets: 6


In [10]:
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

final_models = {}

print("TRAINING FINAL PROTOTYPE")
print("=" * 60)

for label in TARGET_COLUMNS:

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(final_preprocessor)
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ]
    )

    model.fit(
        X,
        y[label]
    )

    final_models[label] = model

    print(
        f"Trained: {label}"
    )

print(
    "\nModels trained:",
    len(final_models)
)

TRAINING FINAL PROTOTYPE
Trained: hypothermia_guidance
Trained: heat_illness_guidance
Trained: heavy_rain_guidance
Trained: strong_wind_guidance
Trained: water_safety_guidance
Trained: remote_area_emergency_guidance

Models trained: 6


In [11]:
prediction_check = pd.DataFrame(
    index=model_data.index
)

for label in TARGET_COLUMNS:

    prediction_check[label] = (
        final_models[label]
        .predict(X)
    )

print("FINAL MODEL SANITY CHECK")
print("=" * 60)

print(
    "Prediction shape:",
    prediction_check.shape
)

print(
    "Missing predictions:",
    prediction_check.isna().sum().sum()
)

invalid_predictions = 0

for label in TARGET_COLUMNS:

    invalid_predictions += (
        ~prediction_check[label]
        .isin([0, 1])
    ).sum()

print(
    "Invalid predictions:",
    invalid_predictions
)

print("\nPredicted positives:")

print(
    prediction_check.sum()
)

FINAL MODEL SANITY CHECK
Prediction shape: (557, 6)
Missing predictions: 0
Invalid predictions: 0

Predicted positives:
hypothermia_guidance              146
heat_illness_guidance             283
heavy_rain_guidance               257
strong_wind_guidance              206
water_safety_guidance             170
remote_area_emergency_guidance    236
dtype: int64


## Mobile Deployment Architecture

The selected Logistic Regression prototype is intended to support lightweight
offline inference.

The proposed application workflow is:

**1. Detect hiking context**

The mobile application obtains the user's location while network connectivity
is available.

**2. Identify nearby hiking context**

The location is associated with available trail, terrain, environmental,
emergency-access, and communication-remoteness information.

**3. Construct model input**

The required contextual features are prepared using the same feature schema
used during model development.

**4. Predict contextual resources**

The multi-label recommendation model determines which additional offline
guidance categories are relevant to the detected hiking context.

**5. Add mandatory safety resources**

The application always includes mandatory resources independently of ML:

- offline map;
- general first-aid guidance;
- emergency contact information.

**6. Build offline package**

The application combines mandatory resources with predicted contextual
resources.

**7. Download before connectivity loss**

While network connectivity is still available, the user is prompted to
download the recommended offline package.

**8. Offline access**

During the hike, downloaded information remains locally available without
requiring network connectivity.

The ML model therefore controls contextual personalization, while essential
safety information remains protected by deterministic application logic.

In [12]:
import joblib

MODEL_ARTIFACT = (
    MODELS_DIR
    / "offline_resource_logistic_prototype.joblib"
)

model_package = {
    "models": final_models,
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_columns": TARGET_COLUMNS,
    "model_type": "LogisticRegression",
    "training_rows": len(X),
    "version": "prototype_v1"
}

joblib.dump(
    model_package,
    MODEL_ARTIFACT
)

print("MODEL EXPORT")
print("=" * 60)

print(
    "Saved:",
    MODEL_ARTIFACT.exists()
)

print(
    "Path:",
    MODEL_ARTIFACT
)

MODEL EXPORT
Saved: True
Path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/models/offline_resource_logistic_prototype.joblib


In [13]:
model_size_bytes = (
    MODEL_ARTIFACT.stat().st_size
)

model_size_kb = (
    model_size_bytes / 1024
)

model_size_mb = (
    model_size_bytes
    / (1024 ** 2)
)

print("MODEL SIZE")
print("=" * 60)

print(
    f"Bytes: {model_size_bytes:,}"
)

print(
    f"Kilobytes: {model_size_kb:.2f} KB"
)

print(
    f"Megabytes: {model_size_mb:.4f} MB"
)

MODEL SIZE
Bytes: 31,047
Kilobytes: 30.32 KB
Megabytes: 0.0296 MB


In [14]:
loaded_package = joblib.load(
    MODEL_ARTIFACT
)

print("MODEL RELOAD VALIDATION")
print("=" * 60)

print(
    "Model type:",
    loaded_package["model_type"]
)

print(
    "Version:",
    loaded_package["version"]
)

print(
    "Training rows:",
    loaded_package["training_rows"]
)

print(
    "Number of models:",
    len(
        loaded_package["models"]
    )
)

print(
    "Number of features:",
    len(
        loaded_package["feature_columns"]
    )
)

print(
    "Number of targets:",
    len(
        loaded_package["target_columns"]
    )
)

MODEL RELOAD VALIDATION
Model type: LogisticRegression
Version: prototype_v1
Training rows: 557
Number of models: 6
Number of features: 35
Number of targets: 6


In [15]:
reload_predictions = pd.DataFrame(
    index=X.index
)

for label in TARGET_COLUMNS:

    reload_predictions[label] = (
        loaded_package[
            "models"
        ][label].predict(X)
    )

predictions_identical = (
    reload_predictions[
        TARGET_COLUMNS
    ].to_numpy()
    ==
    prediction_check[
        TARGET_COLUMNS
    ].to_numpy()
).all()

print("SERIALIZATION VALIDATION")
print("=" * 60)

print(
    "Predictions identical after reload:",
    predictions_identical
)

print(
    "Missing predictions:",
    reload_predictions.isna().sum().sum()
)

SERIALIZATION VALIDATION
Predictions identical after reload: True
Missing predictions: 0


In [16]:
import json

MODEL_METADATA_FILE = (
    MODELS_DIR
    / "offline_resource_logistic_prototype_metadata.json"
)

model_metadata = {
    "model_name":
        "offline_resource_logistic_prototype",

    "version":
        "prototype_v1",

    "algorithm":
        "Logistic Regression",

    "problem_type":
        "multi-label classification",

    "training_rows":
        int(len(X)),

    "input_feature_count":
        int(len(MODEL_FEATURE_COLUMNS)),

    "target_count":
        int(len(TARGET_COLUMNS)),

    "targets":
        TARGET_COLUMNS,

    "random_split_macro_f1":
        0.9444,

    "grouped_mean_macro_f1":
        0.2944,

    "grouped_mean_hamming_loss":
        0.2762,

    "model_size_bytes":
        int(model_size_bytes),

    "mandatory_resources": [
        "offline_map",
        "general_first_aid",
        "emergency_contact_info"
    ],

    "limitations": [
        "Targets are rule-derived.",
        "Historical weather context uses five Knuckles reference locations.",
        "Grouped evaluation shows limited unseen-region generalization.",
        "The joblib artifact is a research artifact and is not directly executable by Flutter."
    ]
}

with open(
    MODEL_METADATA_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=2
    )

print("MODEL METADATA")
print("=" * 60)

print(
    "Saved:",
    MODEL_METADATA_FILE.exists()
)

print(
    "Path:",
    MODEL_METADATA_FILE
)

MODEL METADATA
Saved: True
Path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/models/offline_resource_logistic_prototype_metadata.json


## Prototype Deployment Status

The final research prototype consists of six Logistic Regression classifiers,
one for each contextual offline-resource recommendation target.

The complete preprocessing and classification pipelines have been serialized
as a research artifact and successfully reloaded to verify reproducibility.

The serialized Joblib artifact is intended for Python-side research,
reproducibility, and model inspection. It is not directly executable inside
the Flutter application.

For mobile deployment, the trained preprocessing parameters and Logistic
Regression coefficients must be represented in a mobile-compatible inference
implementation or converted to an appropriate supported format.

This separation prevents the research artifact from being incorrectly treated
as a production mobile model.

The final mobile system will continue to use a hybrid architecture:

- deterministic mandatory safety resources;
- lightweight contextual ML recommendations;
- locally stored offline resource content;
- network-assisted context acquisition and package download before expected
  connectivity loss.

In [17]:
required_artifacts = {
    "Research model": MODEL_ARTIFACT,
    "Model metadata": MODEL_METADATA_FILE,
    "Feature dataset": FEATURE_FILE,
    "Label dataset": LABEL_FILE
}

print("FINAL ARTIFACT VALIDATION")
print("=" * 60)

all_artifacts_exist = True

for name, path in required_artifacts.items():

    exists = path.exists()

    print(
        f"{name}: {exists}"
    )

    if not exists:
        all_artifacts_exist = False

print()
print(
    "All required artifacts available:",
    all_artifacts_exist
)

FINAL ARTIFACT VALIDATION
Research model: True
Model metadata: True
Feature dataset: True
Label dataset: True

All required artifacts available: True
